# Check the report - EP_0002

Read the claims beside the evidence, note anything wrong, edit the copy below, then save it to public.
The original model response stays in the attempt folder. No model calls here.

In [1]:
import copy
import json
import sys
from decimal import Decimal
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/packets").is_dir())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, save, mask, assert_masked, check_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
attempt_id = "1453e4303c4b4412bffb36eb1a768020"
attempt_folder = folder / "private/attempts" / attempt_id
attempt = read(attempt_folder / "attempt.json")
sent_packet = json.loads(attempt["request"]["messages"][1]["content"])["evidence_packet"]
original_report = read(attempt_folder / "candidate.json")
aliases = read(folder / "private/aliases.json")
packet = mask(sent_packet, aliases)
report = copy.deepcopy(original_report)
print("Packet:", packet["packet_id"], "| Model:", attempt["request"]["model"])
print("Packet masking changed:", packet != sent_packet)

Packet: EP_0002 | Model: google/gemini-2.5-flash
Packet masking changed: True


In [2]:
evidence = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
for claim in report["claims"]:
    print("\nCLAIM", claim["claim_id"], claim["text"])
    for evidence_id in claim["evidence_ids"]:
        print("EVIDENCE", evidence_id, evidence.get(evidence_id, "MISSING"))
print("\nRationale:", report["rationale"])


CLAIM C1 Revenue increased by 14.93% from $245,122 million in fiscal year 2024 to $281,724 million in fiscal year 2025.
EVIDENCE EP_0002_E01 SUMMARY RESULTS OF OPERATIONS
 
(In millions, except percentages and per share amounts)  2025   2024   
Percentage
Change  
          
          
Revenue  $ 281,724  $ 245,122    15%  
Gross margin   193,893   171,008    13%  
Operating income   128,528   109,433    17%  
Net income   101,832   88,136    16%  
Diluted earnings per share   13.64   11.80    16%

CLAIM C2 Operating income increased by 17.45% from $109,433 million in fiscal year 2024 to $128,528 million in fiscal year 2025.
EVIDENCE EP_0002_E01 SUMMARY RESULTS OF OPERATIONS
 
(In millions, except percentages and per share amounts)  2025   2024   
Percentage
Change  
          
          
Revenue  $ 281,724  $ 245,122    15%  
Gross margin   193,893   171,008    13%  
Operating income   128,528   109,433    17%  
Net income   101,832   88,136    16%  
Diluted earnings per share   13.6

## Numbers

Compare the claimed amount or percentage with the calculation. Also read the prose above:
these checks only cover the structured assertion, not every number in a sentence.

In [3]:
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        print(claim["claim_id"], "Check against the excerpt above.")
        continue
    facts = [f for f in packet["facts"] if f["metric"] == spec["metric"]
             and f["evidence_id"] in claim["evidence_ids"]]
    periods = [spec["period"]] if spec["kind"] == "numeric" else [spec["from_period"], spec["to_period"]]
    matches = [[f for f in facts if f["period"] == period] for period in periods]
    if any(len(found) != 1 for found in matches):
        print(claim["claim_id"], "No exact fact match; check manually.")
        continue
    values = [Decimal(found[0]["value"]) for found in matches]
    if spec["kind"] == "percentage_change" and values[0] == 0:
        print(claim["claim_id"], "Cannot calculate percentage change from zero.")
        continue
    expected = values[0] if spec["kind"] == "numeric" else (values[1] - values[0]) / values[0] * 100
    print(claim["claim_id"], "claimed:", spec["value"], "calculated:", expected,
          "rounded:", expected.quantize(Decimal("0.01")))
pprint(check_report(Packet.model_validate(packet), Report.model_validate(report)))

C1 claimed: 14.93 calculated: 14.93215623240671991906071262 rounded: 14.93
C2 claimed: 17.45 calculated: 17.44903274149479590251569453 rounded: 17.45
C3 claimed: 13.38 calculated: 13.38241485778443113772455090 rounded: 13.38
C4 claimed: 69 calculated: 69 rounded: 69.00
C5 Check against the excerpt above.
{'errors': [],
 'human_review': ['C5: check factual support and reasoning manually',
                  'Check prose against structured assertions, neutral tone, '
                  'and material counter-evidence']}


## Notes / feedback

- C1: revenue growth is 14.93%. C2: operating-income growth is 17.45%. C3: gross-margin amount growth is 13.38%. All match the source amounts and years in E01.
- C4: the cloud gross-margin percentage fell to 69%. The explanation matches E02. This is different from growth in the gross-margin dollar amount.
- C5: the $3.8 billion expense increase, quoted 6%, and competition risks match E02/E03. The 6% is the filing's rounded figure, not a new calculation.
- The original excerpts match PDF pages 40 and 17. The PDF hash matches the source record.
- The report includes cloud-margin pressure, higher expenses and competition. HOLD is reasonable from this limited evidence.
- Earlier feedback corrected the growth calculations and restored the cloud-margin decline. The kept candidate already includes these fixes.
- Fixed the line-wrapped 'More Personal Computing' name in the packet to SEGMENT_03. Only the masking changed; the raw request is kept as it was. The report was checked against the corrected packet, not regenerated.
- Made C5 and the rationale clearer that 6% is quoted from the filing. No financial values changed.
- Decision: ready to use as the clean report.

In [4]:
report["claims"][4]["text"] = report["claims"][4]["text"].replace(
    "by $3.8 billion or 6%", "by $3.8 billion (the filing reports rounded growth of 6%)")
report["rationale"] = report["rationale"].replace(
    "by $3.8 billion or 6%", "by $3.8 billion (the filing reports rounded growth of 6%)")

pprint(report, sort_dicts=False)

{'schema_version': '0.1',
 'report_id': 'R_1453e4303c4b4412bffb36eb1a768020',
 'recommendation': 'HOLD',
 'claims': [{'claim_id': 'C1',
             'text': 'Revenue increased by 14.93% from $245,122 million in '
                     'fiscal year 2024 to $281,724 million in fiscal year '
                     '2025.',
             'evidence_ids': ['EP_0002_E01'],
             'check_spec': {'kind': 'percentage_change',
                            'metric': 'revenue',
                            'from_period': {'label': 'FY2024',
                                            'start': '2023-07-01',
                                            'end': '2024-06-30'},
                            'to_period': {'label': 'FY2025',
                                          'start': '2024-07-01',
                                          'end': '2025-06-30'},
                            'value': '14.93',
                            'unit': 'percent'}},
            {'claim_id': 'C2',
             'tex

## Save the final copy

Set `ready = True` after the checks and edits are finished. This writes the final report and
masked packet to public. It does not change the raw attempt or call a model.
Save this notebook in `private/reviews/` using `<packet_id>_review.ipynb`.
One notebook contains the check, feedback, edits and output; no separate review JSON is needed.

The report is saved as `public/reports/<packet_id>/clean.json`.

In [5]:
ready = True
if ready:
    final_report = Report.model_validate(report)
    final_packet = Packet.model_validate(packet)
    checks = check_report(final_packet, final_report)
    assert not checks["errors"], checks["errors"]
    assert_masked(report, aliases)
    assert_masked(packet, aliases)
    save(folder / "public/evidence_packets" / (packet["packet_id"] + ".json"), final_packet.model_dump(mode="json"))
    output = folder / "public/reports" / packet["packet_id"] / "clean.json"
    save(output, final_report.model_dump(mode="json"))
    print("Saved:", output)
else:
    print("Not saved yet.")

Saved: C:\Users\Burthus\repos\agent_test\data\benchmark\public\reports\EP_0002\clean.json
